# Automated GRPO Threshold Ladder Colab Runner

Automated Colab notebook for the single-session post-refactor GRPO threshold ladder.

This notebook:
- bootstraps the repo and checkpoints like `chess_model_run_git.ipynb`
- runs the three approved Stage 1 probes sequentially
- evaluates movement/stability/eval signal using WandB and local summaries
- selects the Stage 2 midpoint bracket automatically
- falls back to the best Stage 1 run if Stage 2 is inconclusive
- uses the remaining Colab session time for a final confirmation run


In [ ]:
#@title Runtime Parameters
REPO_URL = "https://github.com/noamdwc/grpo_chess.git"  #@param {type:"string"}
REPO_REF = "search_refactor"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/data/grpo-chess"  #@param {type:"string"}
BASE_CHECKPOINT_PATH = "/content/drive/MyDrive/data/grpo-chess/base/9M.pt"  #@param {type:"string"}
RUN_NAME_PREFIX = "grpo-threshold-ladder"  #@param {type:"string"}
USE_WANDB = True  #@param {type:"boolean"}
WANDB_API_KEY = ""  #@param {type:"string"}
SESSION_HOURS = 25.0  #@param {type:"number"}
ESTIMATED_MIN_PER_EPOCH = 2.80  #@param {type:"number"}
STAGE3_TARGET_EPOCHS = 300  #@param {type:"integer"}
STAGE3_MIN_EPOCHS = 200  #@param {type:"integer"}
SESSION_RESERVE_MINUTES = 30  #@param {type:"integer"}


In [ ]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("This notebook is intended to run in Google Colab.")

from google.colab import drive
drive.mount("/content/drive")

repo = Path("/content/grpo_chess")
os.chdir("/content")
if repo.exists():
    shutil.rmtree(repo)

!git clone {REPO_URL} /content/grpo_chess
%cd /content/grpo_chess
!git checkout {REPO_REF}
!git submodule update --init --recursive

if str(repo) not in sys.path:
    sys.path.append(str(repo))

drive_root = Path(DRIVE_ROOT)
drive_root.mkdir(parents=True, exist_ok=True)
runtime_dir = drive_root / "colab_runtime"
runtime_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")


In [ ]:
import os
import signal
from pathlib import Path

deps_ready = Path("/tmp/grpo_reasoning_colab_deps_ready")
if not deps_ready.exists():
    %pip install -q --upgrade pip setuptools wheel
    !grep -vE '^numpy==' requirements.txt > /tmp/requirements-colab.txt
    %pip install -q -r /tmp/requirements-colab.txt
    !apt-get -qq update
    !apt-get -qq install -y stockfish
    !COLAB=1 bash scripts/setup_distill_deps.sh --checkpoint 9M --skip-checkpoint
    %pip install -q --force-reinstall --no-cache-dir pillow
    deps_ready.touch()
    print("Dependencies installed. Restarting runtime to load fresh binary modules...")
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("Dependencies already installed for this runtime.")


In [ ]:
import json
import subprocess
import urllib.request
import yaml
import zipfile
from pathlib import Path

base_checkpoint = Path(BASE_CHECKPOINT_PATH).expanduser()
base_checkpoint.parent.mkdir(parents=True, exist_ok=True)

searchless_ckpt_root = drive_root / "searchless_checkpoints"
searchless_ckpt_root.mkdir(parents=True, exist_ok=True)
repo_searchless_ckpt_root = repo / "searchless_chess" / "checkpoints"
repo_searchless_ckpt_root.mkdir(parents=True, exist_ok=True)
download_url = "https://storage.googleapis.com/searchless_chess/checkpoints/9M.zip"
zip_path = searchless_ckpt_root / "9M.zip"
nine_m_dir = searchless_ckpt_root / "9M"
teacher_download_url = "https://storage.googleapis.com/searchless_chess/checkpoints/136M.zip"
teacher_zip_path = searchless_ckpt_root / "136M.zip"
teacher_ckpt_dir = searchless_ckpt_root / "136M"

if not base_checkpoint.exists():
    if not nine_m_dir.exists():
        print("Downloading", download_url)
        urllib.request.urlretrieve(download_url, zip_path)
        print("Extracting", zip_path)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(searchless_ckpt_root)
        if zip_path.exists():
            zip_path.unlink()
    else:
        print("Reusing existing 9M checkpoint directory:", nine_m_dir)

    expected_orbax = nine_m_dir / "6400000" / "params" / "checkpoint"
    if not expected_orbax.exists():
        raise FileNotFoundError(f"Missing expected Orbax checkpoint file: {expected_orbax}")

    print("Converting downloaded 9M checkpoint to DM-port format:", base_checkpoint)
    conversion = subprocess.run(
        [
            sys.executable,
            "-m",
            "src.dm_port.convert_jax",
            "--model",
            "9M",
            "--checkpoint-dir",
            str(searchless_ckpt_root),
            "--out",
            str(base_checkpoint),
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if conversion.stdout:
        print("Converter stdout:
", conversion.stdout)
    if conversion.stderr:
        print("Converter stderr:
", conversion.stderr)
    if conversion.returncode != 0:
        raise RuntimeError(
            f"DM checkpoint conversion failed with exit code {conversion.returncode}. "
            f"See printed converter stdout/stderr above."
        )

if not teacher_ckpt_dir.exists():
    print("Downloading", teacher_download_url)
    urllib.request.urlretrieve(teacher_download_url, teacher_zip_path)
    print("Extracting", teacher_zip_path)
    with zipfile.ZipFile(teacher_zip_path, "r") as zf:
        zf.extractall(searchless_ckpt_root)
    if teacher_zip_path.exists():
        teacher_zip_path.unlink()
else:
    print("Reusing existing 136M checkpoint directory:", teacher_ckpt_dir)

expected_teacher_orbax = teacher_ckpt_dir / "6400000" / "params" / "checkpoint"
if not expected_teacher_orbax.exists():
    raise FileNotFoundError(f"Missing expected 136M Orbax checkpoint file: {expected_teacher_orbax}")

repo_teacher_ckpt = repo_searchless_ckpt_root / "136M"
if repo_teacher_ckpt.exists() or repo_teacher_ckpt.is_symlink():
    repo_teacher_ckpt.unlink()
repo_teacher_ckpt.symlink_to(teacher_ckpt_dir, target_is_directory=True)
print("136M teacher checkpoint ready at:", repo_teacher_ckpt)


In [ ]:
import copy
import dataclasses
import math
import os
import time
from pathlib import Path

import torch
import wandb
import yaml

if USE_WANDB:
    from google.colab import userdata

    if WANDB_API_KEY.strip():
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY.strip()
    else:
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    !wandb login
else:
    os.environ["WANDB_DISABLED"] = "true"

from src.colab_experiment_ladder import (
    RunAssessment,
    choose_confirmation_source,
    select_best_stage1_run,
    select_stage2_bracket,
)
from src.configs.config_loader import load_experiment_config
from src.train_self_play import train as grpo_train
import src.trainer as trainer_module

STAGE_1_CONFIGS = {
    "A1": "grpo_colab_probe_a1_lr3e6_kl5e4.yaml",
    "A2": "grpo_colab_probe_a2_lr5e6_kl3e4.yaml",
    "A3": "grpo_colab_probe_a3_lr8e6_kl1e4.yaml",
}
BRACKET_CONFIGS = {
    "grpo_colab_bracket_b1_mid_low.yaml": "grpo_colab_bracket_b1_mid_low.yaml",
    "grpo_colab_bracket_b1_mid_high.yaml": "grpo_colab_bracket_b1_mid_high.yaml",
}
RUN_SUMMARY_PATH = runtime_dir / f"{RUN_NAME_PREFIX}-summary.json"
SESSION_START_TS = time.time()


def _read_yaml(path: Path):
    return yaml.safe_load(path.read_text())


def _runtime_config_path(run_label: str) -> Path:
    return runtime_dir / f"{run_label}.yaml"


def _run_root(run_label: str) -> Path:
    root = drive_root / "runs" / run_label
    root.mkdir(parents=True, exist_ok=True)
    return root


def _build_runtime_config(config_filename: str, run_label: str, num_epochs: int | None = None) -> Path:
    config_path = Path("src/configs") / config_filename
    config_data = copy.deepcopy(_read_yaml(config_path))
    config_data["model"]["base_checkpoint"] = str(base_checkpoint)
    config_data["rival"]["frozen_dm_9m"]["checkpoint_path"] = str(base_checkpoint)
    config_data["training"]["checkpoint_dir"] = str(_run_root(run_label))
    config_data["training"]["use_wandb"] = bool(USE_WANDB)
    if num_epochs is not None:
        config_data["training"]["num_epochs"] = int(num_epochs)
    runtime_config_path = _runtime_config_path(run_label)
    runtime_config_path.write_text(yaml.safe_dump(config_data, sort_keys=False))
    print("Runtime config:", runtime_config_path)
    print(yaml.safe_dump(config_data, sort_keys=False))
    return runtime_config_path


def _latest_wandb_summary() -> dict:
    latest = Path("wandb/latest-run/files/wandb-summary.json")
    if latest.exists():
        return json.loads(latest.read_text())
    return {}


def _find_api_run(project: str, run_name: str):
    api = wandb.Api()
    entity = os.environ.get("WANDB_ENTITY") or getattr(api, "default_entity", None)
    if not entity:
        return None
    try:
        runs = api.runs(f"{entity}/{project}", per_page=50, order="-created_at")
    except Exception as exc:
        print("WandB API lookup failed:", exc)
        return None
    for run in runs:
        display_name = getattr(run, "display_name", None) or getattr(run, "name", None)
        if display_name == run_name or getattr(run, "name", None) == run_name:
            return run
    return None


def _assess_run(run_name: str, project: str) -> RunAssessment:
    summary = _latest_wandb_summary()
    api_run = _find_api_run(project, run_name) if USE_WANDB else None
    if api_run is not None:
        summary = {**summary, **dict(getattr(api_run, "summary", {}))}

    ratio = float(summary.get("train/ratio_step", 1.0))
    clip_fraction = float(summary.get("train/clip_fraction_step", 0.0))
    ppo_loss = float(summary.get("train/ppo_loss_step", 0.0))
    kl_div = float(summary.get("train/kl_divergence_step", 0.0))
    eval_score = float(summary.get("eval_stockfish/score", 0.0))

    movement_score = (
        abs(ratio - 1.0) * 2000.0
        + clip_fraction * 5.0
        + abs(ppo_loss) * 1000.0
        + abs(kl_div) * 500.0
    )
    inert = abs(ratio - 1.0) < 0.001 and clip_fraction < 0.01 and abs(ppo_loss) < 5e-4 and abs(kl_div) < 5e-4
    unstable = clip_fraction > 0.5 or abs(ratio - 1.0) > 0.2 or abs(kl_div) > 0.02
    stable = not unstable
    result = RunAssessment(
        name=run_name,
        movement_score=movement_score,
        stable=stable,
        eval_score=eval_score,
        inert=inert,
        unstable=unstable,
        config_name=run_name,
    )
    print("Assessment for", run_name, dataclasses.asdict(result))
    return result


def run_experiment(stage_name: str, config_filename: str, *, num_epochs: int | None = None) -> RunAssessment:
    run_label = f"{RUN_NAME_PREFIX}-{stage_name.lower()}"
    runtime_config_path = _build_runtime_config(config_filename, run_label, num_epochs=num_epochs)
    cfg = load_experiment_config(runtime_config_path)
    print("Resolved base checkpoint:", cfg.model.base_checkpoint)
    print("Resolved checkpoint dir:", cfg.training.checkpoint_dir)
    print("Resolved eval cadence (epochs):", cfg.grpo.eval_every_n_epochs)
    print("Expected WandB eval metrics: eval_stockfish_init/*, eval_stockfish/*")

    original_generate_run_name = trainer_module.generate_run_name
    trainer_module.generate_run_name = lambda project="chess-grpo": run_label
    wandb.finish()
    start_ts = time.time()
    try:
        grpo_train(config_path=str(runtime_config_path), resume_from_checkpoint=None)
    finally:
        trainer_module.generate_run_name = original_generate_run_name
        wandb.finish()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    elapsed_minutes = (time.time() - start_ts) / 60.0
    print(f"Run {run_label} completed in {elapsed_minutes:.2f} minutes")
    return _assess_run(run_label, cfg.training.wandb_project)


def _remaining_confirmation_epochs() -> int:
    elapsed_hours = (time.time() - SESSION_START_TS) / 3600.0
    remaining_hours = max(0.0, SESSION_HOURS - elapsed_hours - (SESSION_RESERVE_MINUTES / 60.0))
    estimated_epochs = int((remaining_hours * 60.0) / max(ESTIMATED_MIN_PER_EPOCH, 0.1))
    return max(STAGE3_MIN_EPOCHS, min(STAGE3_TARGET_EPOCHS, estimated_epochs))


def _write_summary(payload: dict) -> None:
    RUN_SUMMARY_PATH.write_text(json.dumps(payload, indent=2, sort_keys=True))
    print("Wrote ladder summary to", RUN_SUMMARY_PATH)


In [ ]:
from dataclasses import asdict, replace

stage1_results = {}
for stage_name, config_filename in STAGE_1_CONFIGS.items():
    stage1_results[stage_name] = run_experiment(stage_name, config_filename)

best_stage1 = select_best_stage1_run(stage1_results)
print("Best Stage 1 run:", best_stage1)

stage2_config = select_stage2_bracket(stage1_results, best_stage1=best_stage1)
print("Selected Stage 2 bracket:", stage2_config)
stage2_result = run_experiment("B1", stage2_config, num_epochs=100)

best_stage1_assessment = stage1_results[best_stage1]
stage2_delta_movement = stage2_result.movement_score - best_stage1_assessment.movement_score
stage2_delta_eval = stage2_result.eval_score - best_stage1_assessment.eval_score
stage2_inconclusive = (
    stage2_result.unstable
    or not stage2_result.stable
    or (stage2_delta_movement < 0.05 and stage2_delta_eval < 0.01)
)
stage2_result = replace(stage2_result, inconclusive=stage2_inconclusive)

confirmation_source = choose_confirmation_source(best_stage1_assessment, stage2_result)
if confirmation_source == "B1":
    confirmation_config = stage2_config
else:
    confirmation_config = STAGE_1_CONFIGS[confirmation_source]

remaining_epochs = _remaining_confirmation_epochs()
print("Stage 3 source:", confirmation_source)
print("Stage 3 config:", confirmation_config)
print("Stage 3 epochs:", remaining_epochs)

confirmation_result = run_experiment("C1", confirmation_config, num_epochs=remaining_epochs)

summary_payload = {
    "stage1_results": {name: asdict(result) for name, result in stage1_results.items()},
    "best_stage1": best_stage1,
    "stage2_config": stage2_config,
    "stage2_result": asdict(stage2_result),
    "confirmation_source": confirmation_source,
    "confirmation_config": confirmation_config,
    "remaining_confirmation_epochs": remaining_epochs,
    "confirmation_result": asdict(confirmation_result),
}
_write_summary(summary_payload)
summary_payload
